# E_supplementary_centroids

See the repository README for inputs, environment variables, and execution status. Except where listed in `REVISION_VALIDATION.md`, saved outputs are historical and were not rerun for this revision. The two A320 bias algorithms are instructor-supplied; assistance and supplementary analysis are disclosed in the report appendix.

In [1]:
from pathlib import Path
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image
%matplotlib inline
ROOT = Path.cwd().parent
DATA = ROOT / 'data'
OUT = ROOT / 'results'
OUT.mkdir(exist_ok=True)
RAW = Path(os.environ.get('ASTRO320W_DATA', str(ROOT / 'raw')))
BIAS = Path(os.environ.get('ASTRO320W_BIAS', str(ROOT / 'raw' / 'Bias_Frames')))


In [2]:
import numpy as np
from scipy.optimize import least_squares
from astropy.io import fits
from pathlib import Path
import csv
root=RAW/'Exercise E/EXERCISE E'
def fit(im,x,y):
 xi,yi=int(round(x)),int(round(y));v=im[yi-8:yi+9,xi-8:xi+9].astype(float);yy,xx=np.mgrid[yi-8:yi+9,xi-8:xi+9];bg=np.median(v)
 def model(p):
  a,cx,cy,sx,sy,c,dx,dy=p
  return a*np.exp(-.5*(((xx-cx)/sx)**2+((yy-cy)/sy)**2))+c+dx*(xx-xi)+dy*(yy-yi)
 p=[max(v.max()-bg,1),x,y,2.5,2.5,bg,0,0]
 r=least_squares(lambda p:(model(p)-v).ravel(),p,bounds=([0,x-4,y-4,.8,.8,0,-100,-100],[1e6,x+4,y+4,8,8,1e5,100,100]),loss='soft_l1',f_scale=20)
 return r.x,np.sqrt(np.mean((model(r.x)-v)**2)),v.max()
rows=[]
for n in [18,19,20]:
 im=fits.getdata(root/f'CCD Image {n}.fit'); yy,xx=np.unravel_index(np.argmax(im[595:625,670:700]),(30,30));xp,yp=xx+670,yy+595
 A,er,peak=fit(im,xp,yp)
 # Bright primary: positive-background-subtracted centroid in a fixed 12-pixel radius.
 cx,cy=683.,609.
 for k in range(5):
  yy,xx=np.mgrid[int(cy)-16:int(cy)+17,int(cx)-16:int(cx)+17];v=im[yy,xx].astype(float);r=np.hypot(xx-cx,yy-cy);bg=np.median(v[r>13]);weights=np.maximum(v-bg,0)*(r<=12);cx=np.sum(xx*weights)/weights.sum();cy=np.sum(yy*weights)/weights.sum()
 A[1:3]=cx,cy
 B,err,pk=fit(im,A[1]+26.4,A[2]-57.6);d=np.hypot(A[1]-B[1],A[2]-B[2]);print(n,A[1:5],B[1:5],d,80.4/d,'rms',er,err,'peak',peak,pk)
 rows.append([n,A[1],1528-A[2],B[1],1528-B[2],d,80.4/d,pk,err])
with open(OUT/'HR7310_supplementary.csv','w') as f:
 w=csv.writer(f);w.writerow(['original_image','x1_IJ','y1_IJ','x2_IJ','y2_IJ','distance_pixels','scale_80p4_arcsec','companion_peak','companion_fit_rms']);w.writerows(rows)
print('means',np.mean(np.array(rows)[:,5:7],axis=0),'SEM',np.std(np.array(rows)[:,5:7],axis=0,ddof=1)/np.sqrt(3))


18 [683.15457558 608.56462348   3.08495684   3.19623795] [707.69965749 548.32301944   1.42227915   1.96337878] 65.05007227620298 1.2359709557065692 rms 7110.260620750573 25.394015512614736 peak 47924.0 355.0
19 [682.22168668 609.754817     3.29522464   3.27535668] [706.96573513 549.21035859   1.51557191   1.19706111] 65.40565248335521 1.22925155467963 rms 6903.0094681182 25.501913037228675 peak 48092.0 308.0
20 [682.11072925 609.68254687   3.28992979   3.43139212] [706.68021183 549.06864265   1.34428921   1.06356799] 65.40416544993727 1.2292795030240253 rms 6616.289630320725 26.094037088518565 peak 47886.0 352.0
means [65.28663007  1.23150067] SEM [0.11827968 0.00223516]


In [3]:
import numpy as np
from scipy.optimize import least_squares
from astropy.io import fits
from pathlib import Path
import csv
root=RAW/'Exercise E/EXERCISE E'
def fit(im,x,y):
 xi,yi=int(round(x)),int(round(y));v=im[yi-8:yi+9,xi-8:xi+9].astype(float);yy,xx=np.mgrid[yi-8:yi+9,xi-8:xi+9];bg=np.median(v)
 def model(p):
  a,cx,cy,sx,sy,c,dx,dy=p
  return a*np.exp(-.5*(((xx-cx)/sx)**2+((yy-cy)/sy)**2))+c+dx*(xx-xi)+dy*(yy-yi)
 p=[max(v.max()-bg,1),x,y,2.5,2.5,bg,0,0]
 r=least_squares(lambda p:(model(p)-v).ravel(),p,bounds=([0,x-4,y-4,.8,.8,0,-100,-100],[1e6,x+4,y+4,8,8,1e5,100,100]),loss='soft_l1',f_scale=20)
 return r.x,np.sqrt(np.mean((model(r.x)-v)**2)),v.max()
rows=[]
# Frame 17 excluded: user reports a streak/image artifact.
for n in [15,16]:
 im=fits.getdata(root/f'CCD Image {n}.fit')
 A,er,peak=fit(im,1073.,822.)
 B,err,pk=fit(im,1063.,789.)
 d=np.hypot(A[1]-B[1],A[2]-B[2]);print(n,A[1:3],B[1:3],d,45.6/d,'rms',er,err,'peaks',peak,pk)
 rows.append([n,A[1],1528-A[2],B[1],1528-B[2],d,45.6/d,pk,err])
with open(OUT/'HR6446_supplementary.csv','w') as f:
 w=csv.writer(f);w.writerow(['original_image','x1_IJ','y1_IJ','x2_IJ','y2_IJ','distance_pixels','diagnostic_scale_45p6_arcsec','companion_peak','companion_fit_rms']);w.writerows(rows)


15 [1072.62072913  821.58784953] [1062.37661689  788.8167555 ] 34.33491575304111 1.3280941280877068 rms 566.6889302194918 24.367286486727842 peaks 32339.0 452.0
16 [1073.66623218  820.28733311] [1062.27330968  786.37557801] 35.77437374614581 1.2746554369777825 rms 1786.8267681368986 29.416369981602806 peaks 38656.0 604.0
